# 5. Implementing multilayer neural networks

Now the third core PyTorch component: the deep learning library itself - reusable building blocks
for defining neural networks.

We'll implement a **multilayer perceptron (MLP)**: a fully connected network with two hidden
layers. In PyTorch, you subclass `torch.nn.Module` and define:

- `__init__`: what layers exist
- `forward`: how data flows through them (this *is* the computation graph from notebook 03)

You essentially never implement `backward` yourself - autograd (notebook 03/04) handles it.

```mermaid
flowchart LR
    In["input<br/>(num_inputs)"] --> L1["Linear(num_inputs, 30)"] --> R1["ReLU"]
    R1 --> L2["Linear(30, 20)"] --> R2["ReLU"]
    R2 --> L3["Linear(20, num_outputs)"] --> Out["logits<br/>(num_outputs)"]
```

`self.layers` is exactly this chain, and `forward(x)` is just "run `x` through it" — the diagram
above and the `Sequential(...)` call in `__init__` describe the same thing two ways.

In [1]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits


`torch.nn.Sequential` just chains layers in order so we don't have to call each one by hand in `forward`. Instantiate the model and inspect it:

In [2]:
model = NeuralNetwork(50, 3)
print(model)


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


Count trainable parameters (every `torch.nn.Linear` weight matrix + bias vector where `requires_grad=True`):

In [3]:
num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print("Total number of trainable model parameters:", num_params)


Total number of trainable model parameters: 2213


Inspect a specific layer's weight matrix:

In [4]:
print(model.layers[0].weight)
print(model.layers[0].weight.shape)   # torch.Size([30, 50])


Parameter containing:
tensor([[ 0.0858,  0.0749, -0.0501,  ..., -0.0619,  0.0194,  0.1062],
        [ 0.0414,  0.1313, -0.0439,  ...,  0.1062, -0.1070,  0.0350],
        [-0.1050, -0.0885,  0.0259,  ..., -0.0340, -0.0227,  0.0285],
        ...,
        [ 0.0557,  0.0255, -0.0973,  ..., -0.0298,  0.0786, -0.0052],
        [ 0.0518,  0.0441, -0.1123,  ..., -0.0898, -0.0704, -0.1123],
        [-0.1215, -0.1072,  0.0103,  ..., -0.0953,  0.1412,  0.0543]],
       requires_grad=True)
torch.Size([30, 50])


Weights are initialized with small random numbers (to break symmetry - if every neuron started
identical, they'd all learn the same thing). Seed the RNG for reproducibility:

In [5]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)


Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


## The forward pass

Feed a random input through the (untrained) model:

In [6]:
torch.manual_seed(123)

X = torch.rand((1, 50))
out = model(X)
print(out)


tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


Notice `grad_fn=<AddmmBackward0>` on the output - PyTorch recording that this tensor came from a
matrix-multiply-then-add (`Addmm`) operation, information it needs for backpropagation.

**Inference mode.** If we're only predicting (not training), building the autograd graph wastes
memory and compute. Wrap inference code in `torch.no_grad()`:

In [7]:
with torch.no_grad():
    out = model(X)
print(out)


tensor([[-0.1262,  0.1080, -0.1792]])


**Logits vs. probabilities.** PyTorch models conventionally return raw `logits` (no final
activation), because loss functions like `cross_entropy` fold the softmax in internally for
numerical stability. To get interpretable class probabilities yourself, apply softmax explicitly:

In [8]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)


tensor([[0.3113, 0.3934, 0.2952]])


Roughly-equal probabilities here are expected - the model hasn't been trained yet. Next: how to
feed real, batched data into a model like this for training.